<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mesozoic Labs — Stable-Baselines3 Training

Train one of the registered species through balance, locomotion, and a species-specific simulator task with PPO or SAC.

Pick a behavior in `BEHAVIOR` (a recipe label — `stand`, `walk`, `hunt` — or a deliverable's stage id) and the notebook walks that behavior's chain from the species' stage manifest, root first, within one session: each node REUSES a certified checkpoint from this run or from the trunk run (`TRUNK_FROM`: a pinned run, or under `"auto"` the sibling run the resolve cell selects) where the reuse rule allows, is JUDGED if it was trained but never gated, and is TRAINED from its parent's handoff otherwise (`docs/BEHAVIOR_RECIPES_PLAN.md` §4.7). Later nodes consume the handoff records held in memory; files saved to Drive support analysis, archival and the cross-run reuse of certified ancestors, and the run bundle publishes every certified deliverable of the chain.

Change `SPECIES` in the configuration cell below. Stage budgets, environment settings, and algorithm hyperparameters are loaded from each species' `configs/<species>/stages.toml` manifest (or legacy `stage*.toml` files); those files and the generated species catalog are authoritative. Values differ by species and stage, so this notebook does not duplicate them.

For a smoke test, override the budget deliberately. For a full run, start with the configured values, measure memory and throughput on the target machine, and adjust parallel environments only from observed resource use. The repository does not publish a validated hardware-to-runtime or batch-size table.

## 1. Setup & Installation
`REPO_REF` selects the code installed in Colab. Setup never discards checkout edits and refuses to switch code after repository modules have been imported; restart the runtime when changing revisions. Local notebooks use the current local checkout.


In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os
import sys

REPO_REF = "main"  # @param {"type":"string"}

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Legacy gym (and its shim packages) segfaults under NumPy 2.x on some
    # Colab images; purge it BEFORE the first stable_baselines3 import so
    # nothing can resolve to it. No-op when the packages are absent.
    get_ipython().system("pip uninstall -y -q gym gym-notices shimmy")

    # Install packages only if not already present. Every requirement is
    # QUOTED — an unquoted gymnasium>=0.29.0 made the shell treat ">" as a
    # redirect, installing an unpinned "gymnasium" and writing a junk file
    # named "=0.29.0" — and pinned to the exact tested set (the repo's
    # venv/CI versions) instead of open ranges.
    if any(importlib.util.find_spec(name) is None for name in ("mujoco", "gymnasium", "stable_baselines3", "mediapy", "matplotlib", "scipy", "imageio_ffmpeg")):
        get_ipython().system(
            'pip install -q "mujoco==3.10.0" "gymnasium==1.3.0" "stable-baselines3[extra]==2.9.0" mediapy matplotlib "scipy>=1.10.0" "imageio-ffmpeg>=0.5.1"'
        )
    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not REPO_REF.strip() or REPO_REF.startswith("-"):
        raise ValueError("REPO_REF must name a branch, tag, or commit.")
    fresh_checkout = not repo_dir.exists()
    if fresh_checkout:
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if not (repo_dir / ".git").exists():
        raise RuntimeError(f"{repo_dir} is not a Git checkout; use a fresh runtime or a different directory.")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=repo_dir, check=True)
    requested_commit = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=repo_dir, text=True).strip()
    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
    if fresh_checkout or current_commit != requested_commit:
        if any(name == "environments" or name.startswith("environments.") for name in sys.modules):
            raise RuntimeError("REPO_REF changed after repository modules were imported. Restart the runtime, then rerun setup.")
        if not fresh_checkout and subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip():
            raise RuntimeError("The checkout has local edits. Save them before changing REPO_REF, or use a fresh runtime; no edits were discarded.")
        subprocess.run(["git", "checkout", "--detach", requested_commit], cwd=repo_dir, check=True)
    print(f"Repository ref: {REPO_REF}; commit: {requested_commit}")
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally; using this checkout. REPO_REF selects only the Colab checkout.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "configs/species_manifest.toml").is_file()),
        None,
    )
    if repo_root is None:
        raise RuntimeError("Open this notebook from the mesozoic-labs checkout or its notebooks directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

from environments.shared.config import load_all_stages
from environments.shared.evaluation import evaluate
from environments.shared.stage_manifest import load_stage_manifest, stage_dirname, stage_label

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Select a full species name in `SPECIES`. The notebook resolves it through the same registry as the SB3 CLI. Existing IDs and aliases also work. Config paths and saved runs retain stable IDs such as `trex`.

`BEHAVIOR` selects the deliverable to train (default `"hunt"`: stance → locomotion → behavior; `"stand"` on a species with a recovery stage is stance → recovery, whose frozen gate is then enforced). `TRUNK_FROM` selects the run whose certified ancestors are reused instead of retrained — the target node itself is never reused across runs. `"auto"` (the default, decision D-A25) picks it for you: the run under this species/algorithm log directory whose certified ancestors cover the most of the chain root-first under the §4.2 rule, the greatest run directory name on a tie (the newest for the storage cell's timestamp ids; a custom `RUN_ID` such as `my_run` sorts after every timestamp); the resolve cell prints the choice, the replication each reused node rests on, the runs it refused and why (the first 20; `TRUNK_SELECTION.candidates` holds every run scanned), and any older-interface parent the command-line widen tool could widen. A run whose `provenance.json` names another species, algorithm or backend is refused before the rules run. A run id (or an absolute run directory) pins one; `""` trains every node here. `RETRAIN_FROM` names a chain node that must be trained here together with everything below it even when a certified copy exists; `RUN_LABEL` is a free-text label recorded beside every trained node's hyperparameter digest. `RUN_ID` names the run directory when the storage cell runs: `""` keeps the run this runtime's storage cell resolved last in the tree `SPECIES`, `ALGORITHM` and `QUICK_TEST` select and mints a fresh timestamped run when there is none there (the first pass, or after one of those changed), a new id starts a fresh run in this runtime, and an existing run's id re-enters that run in place to finish an interrupted or `partial` run (section 5); the storage cell prints whether it minted a new run or re-entered one, and runs again after `RUN_ID` changes. A run directory that already records a node is never overwritten: a new variant is a new `RUN_ID`. A run whose bundle is `complete` takes no new node: a session that would train or judge one into it is refused before anything is trained or written (by the resolve cell, or by the chain loop for what the resolve cell cannot predict, such as a node held only as an `ancestors/` record that the trunk no longer certifies), and the refusal names the fresh `RUN_ID` and the `TRUNK_FROM` that reuse its certified nodes instead. A checkpoint trained behind this checkout's policy interface is widened with the command-line `widen_checkpoint` tool into a new run, which this notebook then judges with `RUN_ID` set to that run, `SEED` set to the parent run's seed and `TRUNK_FROM = ""` (the storage cell refuses another `SEED`, and the resolve cell refuses a trunk until the widened root holds a verdict); section 5 has the recipe (`docs/BEHAVIOR_RECIPES_PLAN.md` §4.6; decisions D-C13, D-C14, D-D14).

The six training selections include **Compsognathus Longipes** (anatomical proxy) and **Compsognathus Longipes (Robot)**. Each has separate PPO/SAC configs, normalization statistics and checkpoint identities. Their stages are balance, locomotion and non-contact target reaching. The robot keeps its fixed head and unpowered tail.

Compsognathus policies currently use simulator state and target coordinates with an MLP; the single head camera is available for rendering but is not a policy input. These are initial simulation recipes, not demonstrated walking policies or onboard controllers. Run the zero-action baseline below before tuning; stable standing alone is not evidence of learned locomotion. `QUICK_TEST` checks the pipeline and may stop at a curriculum gate. A quick test trains 50,000 steps per node and writes its run under `<algorithm>_quick_test/` beside the real runs, so `TRUNK_FROM = "auto"` never selects it and it never counts as a seed replicate of a real run; it can still reuse real certified ancestors.

Names come from `configs/species_manifest.toml`; see [species naming](../docs/SPECIES_NAMING.md) and the [Compsognathus training guide](../environments/compsognathus/README.md#training).

In [ ]:
# ===== SPECIES SELECTION =====
from environments.shared.species_names import species_display_name
from environments.shared.species_registry import get_species_config

SPECIES = "Velociraptor Mongoliensis"  # @param ["Velociraptor Mongoliensis", "Tyrannosaurus Rex", "Brachiosaurus Altithorax", "Dibothrosuchus Elaphros", "Compsognathus Longipes", "Compsognathus Longipes (Robot)"]

# Resolve before creating any log directories or provenance records.
SPECIES_CFG = get_species_config(SPECIES)
SPECIES = SPECIES_CFG.species
SPECIES_DISPLAY_NAME = species_display_name(SPECIES)

# ===== Training settings =====
ALGORITHM = "ppo"  # "ppo" or "sac"
N_ENVS = 4  # Number of parallel environments
SEED = 42  # Random seed for reproducibility
VERBOSE = 0  # 0=quiet, 1=progress bar, 2=debug
QUICK_TEST = False  # True = tiny run to verify setup works (50k steps a node, under <algo>_quick_test/: never a trunk)
USE_GOOGLE_DRIVE = True  # Mount Google Drive to save results
AUTO_DISCONNECT = True  # Disconnect the Colab runtime when training halts (gate failure or completion)

# ===== Behavior recipe (BEHAVIOR_RECIPES_PLAN §4.7) =====
BEHAVIOR = "hunt"  # @param ["stand", "walk", "hunt"] {"allow-input":true}
TRUNK_FROM = "auto"  # "auto" (D-A25): the run under this species/algorithm log directory whose certified ancestors cover the most of the chain, the greatest run directory name on a tie (the newest timestamp id), chosen in the resolve cell and printed; a run id or absolute path pins one; "" trains every node here
RETRAIN_FROM = ""  # optional chain node to train here with every node below it (empty = off; D-A19)
RUN_LABEL = ""  # optional free-text label recorded beside each trained node's hyperparameter digest (D-A21)
RUN_ID = ""  # "" = the run this runtime's storage cell resolved last (a fresh timestamped run on the first pass or after SPECIES, ALGORITHM or QUICK_TEST changed); a new id starts a fresh run; an existing run id re-enters that run to finish an interrupted or partial run (section 5)

print(f"Species: {SPECIES_DISPLAY_NAME}")
print(f"Algorithm: {ALGORITHM.upper()}")
print(f"Quick test mode: {QUICK_TEST}")
print(f"Behavior: {BEHAVIOR}")

## 3. Open the run and resolve the chain

Nothing trains here. The storage cell opens this session's run directory (`RUN_ID`) and records its provenance; the resolve cell loads the stage configs, resolves `BEHAVIOR`'s chain, selects the trunk run under `TRUNK_FROM = "auto"` and refuses a session that would write into a `complete` run.

In [ ]:
# ============================================================
# Storage Configuration
# ============================================================
# When USE_GOOGLE_DRIVE is True and running in Colab, logs and
# models are saved to Google Drive so they persist across sessions.
# Otherwise, everything is saved to the local filesystem.

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    LOG_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Logs will be saved to: {LOG_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")
else:
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")

# QUICK_TEST runs live in their own tree, <algo>_quick_test/, beside the real runs:
# the trunk selection scans <algo>/ only and seed-replicate discovery scans RUN_DIR's
# own siblings, so a 50k-step quick test can never become a TRUNK_FROM = "auto"
# parent or a replicate of a real run (quick tests may still count one another).
_runs_root = LOG_BASE / SPECIES / (ALGORITHM.lower() + ("_quick_test" if QUICK_TEST else ""))

# RUN_ID (configuration cell) names this session's run directory. "" re-enters the
# run this cell resolved last in this runtime (_ACTIVE_RUN_ID, the memo every later
# cell reads as this run's id) and mints a fresh timestamped run only when there is
# none, so re-running the configuration cell (which resets RUN_ID to "") never moves
# the run. The memo counts only in the tree it was opened in: after SPECIES,
# ALGORITHM or QUICK_TEST changed, "" mints a fresh run instead of reusing its id in
# another tree. An explicit RUN_ID always wins over the memo and, once the provenance
# accepts it, is memoised in turn: a new id is how to start a fresh run in this
# runtime. An existing run's id re-enters that run in place, to finish an
# interrupted or partial run; certified nodes of an EARLIER run come in through
# TRUNK_FROM (below), never by pointing RUN_ID at that run, and a run whose bundle
# is complete takes no new node (refused before anything is written: by the resolve
# cell, or by the chain loop for a node the resolve cell cannot predict). RUN_DIR
# follows RUN_ID only through this cell: run it again after changing RUN_ID.
if RUN_ID != RUN_ID.strip() or RUN_ID in (".", "..") or (RUN_ID and Path(RUN_ID).name != RUN_ID):
    raise ValueError(
        f"RUN_ID={RUN_ID!r} is not a run id: it names a run directory under "
        f"{_runs_root}, with no path and no surrounding whitespace "
        "(TRUNK_FROM is the knob that also takes a path)"
    )
_memo_dir = globals().get("RUN_DIR")
_memo_id = globals().get("_ACTIVE_RUN_ID", "") if _memo_dir is not None and Path(_memo_dir).parent == _runs_root else ""
_run_id = RUN_ID or _memo_id or datetime.now().strftime("%Y%m%d_%H%M%S")
_run_dir = _runs_root / _run_id

# D-C14: a root widened into this run on the command line (widen_checkpoint
# --to-stage-dir) keeps its parent run's seed, and the provenance minted below
# publishes training_seed = SEED, so any other SEED refuses here: before anything
# is written. RUN_DIR and the memo move only once the provenance accepts the run.
from environments.shared.result_bundle import read_bundle_status, refuse_widened_seed_mismatch

refuse_widened_seed_mismatch(_run_dir, seed=SEED)
if _run_dir.is_dir():
    _held = ", ".join(sorted(path.name for path in _run_dir.iterdir() if path.is_dir())) or "no directory"
    _run_state = f"re-entering run {_run_id!r}: holds {_held}; bundle status {read_bundle_status(_run_dir) or 'none'}"
else:
    _run_state = "new run" + (": RUN_ID names no existing run" if RUN_ID else "")
print(f"Run directory: {_run_dir} ({_run_state})")
_run_dir.mkdir(parents=True, exist_ok=True)

# Capture immutable code, runtime, seed, and plant provenance before training.
# Re-running this cell is safe when the identifying settings are unchanged.
from environments.shared.constants import PUBLICATION_SEED_START
from environments.shared.plant_contract import current_plant_identity
from environments.shared.result_bundle import initialize_result_bundle, validate_result_bundle

CHECKPOINT_SELECTION_SEED = SEED + 1000
EVALUATION_SEED = SEED + 3000
HARDWARE_LABEL = "Google Colab" if IN_COLAB else "local"
PLANT_IDENTITY = current_plant_identity(SPECIES)
PROVENANCE_PATH = initialize_result_bundle(
    _run_dir,
    species=SPECIES,
    algorithm=ALGORITHM,
    backend="stable-baselines3",
    seed=SEED,
    evaluation_seeds=[CHECKPOINT_SELECTION_SEED, EVALUATION_SEED],
    seed_roles={
        "training": SEED,
        "checkpoint_selection_evaluation": CHECKPOINT_SELECTION_SEED,
        "publication_evaluation": EVALUATION_SEED,
        # The 40-seed certification panel (3042-3081) the stance report and the
        # recovery resolver both roll from; the audit binds their evidence to it
        # (BEHAVIOR_RECIPES_PLAN §4.5, decision D-B17).
        "certification_panel": PUBLICATION_SEED_START,
    },
    evaluation_episodes=30,
    parallel_envs=N_ENVS,
    hardware=HARDWARE_LABEL,
    plant_identity=PLANT_IDENTITY.to_dict(),
    run_id=_run_id,
    repository_root=repo_root,
)
# Only a run whose provenance accepted this session becomes this runtime's run: a
# refused RUN_ID (another run's seed or plant) never reaches the memo.
_ACTIVE_RUN_ID, RUN_DIR = _run_id, _run_dir
print(f"Provenance:    {PROVENANCE_PATH}")

# TRUNK_FROM: an earlier run whose certified ancestors the chain loop may reuse
# (BEHAVIOR_RECIPES_PLAN §4.2). A run id resolves under this species/algorithm
# log directory; an absolute path names a run directory anywhere. It must be a
# run whose provenance.json names the SAME species, algorithm and backend, and
# never this run; its bundle status does not matter (complete, partial or
# failed): the chain loop reuses a node only when the §4.2 rule certifies that
# node itself.
from environments.shared.result_bundle import ResultBundleError, canonical_algorithm, load_provenance

TRUNK_DIR = None
AUTO_TRUNK = False
if TRUNK_FROM:
    if TRUNK_FROM == "auto":
        # D-A25: the trunk run is selected in the resolve cell, once the chain and
        # RETRAIN_FROM are known, from the runs beside this one under LOG_BASE.
        AUTO_TRUNK = True
        print(f"Trunk run:     auto (selected in the resolve cell from {LOG_BASE / SPECIES / ALGORITHM.lower()})")
    else:
        TRUNK_DIR = Path(TRUNK_FROM)
        if not TRUNK_DIR.is_absolute():
            TRUNK_DIR = LOG_BASE / SPECIES / ALGORITHM.lower() / TRUNK_FROM
        if TRUNK_DIR.resolve() == RUN_DIR.resolve():
            raise RuntimeError(
                f"TRUNK_FROM={TRUNK_FROM!r} is the run RUN_ID resolved to ({_ACTIVE_RUN_ID!r}); a trunk is an EARLIER "
                "run. To add a node on top of that run, set RUN_ID to a new id no run uses (RUN_ID = \"\" keeps this "
                "run through the memo) and keep TRUNK_FROM"
            )
        try:
            _trunk_provenance = load_provenance(TRUNK_DIR)
        except ResultBundleError as exc:
            raise RuntimeError(
                f"TRUNK_FROM={TRUNK_FROM!r} ({TRUNK_DIR}) is not a run with a provenance.json, so nothing in it "
                f"can be a certified ancestor: {exc}"
            ) from exc
        _trunk_identity = {
            "species": _trunk_provenance.get("species"),
            "algorithm": _trunk_provenance.get("algorithm"),
            "backend": _trunk_provenance.get("backend"),
        }
        _this_identity = {"species": SPECIES, "algorithm": canonical_algorithm(ALGORITHM), "backend": "stable-baselines3"}
        if _trunk_identity != _this_identity:
            raise RuntimeError(
                f"TRUNK_FROM={TRUNK_FROM!r} was trained as {_trunk_identity}, not {_this_identity}; a certified "
                "ancestor must come from the same species, algorithm and backend"
            )
        print(f"Trunk run:     {TRUNK_DIR} (run_id {_trunk_provenance.get('run_id')!r})")

In [ ]:
# Use the shared registry config already resolved in the selection cell.
EnvClass = SPECIES_CFG.env_class

# Stage files and ordering are resolved by each species' stage manifest.
STAGE_CONFIGS = load_all_stages(SPECIES)

# The behavior chain is resolved ONCE, here, through the manifest: BEHAVIOR
# names a deliverable node (a recipe label resolves to the deepest deliverable
# carrying it, an id to itself) and CHAIN is that node's ancestors root-first,
# then the node. The chain loop (section 6) walks CHAIN in this order.
MANIFEST = load_stage_manifest(SPECIES)
TARGET_NODE = MANIFEST.resolve_behavior(BEHAVIOR)
CHAIN = MANIFEST.chain_for(TARGET_NODE.id)
# RETRAIN_FROM (D-A19) names a chain node by its manifest id: it and every
# node below it train here even when a certified copy exists.
RETRAIN_NODE = MANIFEST.resolve(RETRAIN_FROM) if RETRAIN_FROM else None
if RETRAIN_NODE is not None and RETRAIN_NODE not in CHAIN:
    raise RuntimeError(
        f"RETRAIN_FROM={RETRAIN_FROM!r} is not on the chain of behavior {BEHAVIOR!r} "
        f"({[node.id for node in CHAIN]}); name one of its nodes or leave it empty"
    )

env = EnvClass()
try:
    print(f"Species: {SPECIES_DISPLAY_NAME}")
    print(f"Environment: {EnvClass.__name__}")
    print(f"Observation space: {env.observation_space.shape}")
    print(f"Action space: {env.action_space.shape}")
    print(f"Number of actuators: {env.model.nu}")
finally:
    env.close()

print(f"\nBehavior {BEHAVIOR!r} -> deliverable {TARGET_NODE.id!r}; chain: {' -> '.join(node.id for node in CHAIN)}")
if RETRAIN_NODE is not None:
    print(f"RETRAIN_FROM {RETRAIN_NODE.id!r}: it and every node below it train here (no reuse)")
print(f"\n  {'#':>2}  {'id':<12} {'name':<30} {'parent':<12} {'recipe':<8} {'deliverable':<12} gate")
for entry in MANIFEST.stages:
    cfg = STAGE_CONFIGS[entry.reference]
    mark = "*" if entry in CHAIN else " "
    gate = cfg.get("curriculum_kwargs", {}).get("gate_kind", "?")
    print(
        f"{mark} {entry.position:>2}  {entry.id:<12} {cfg['name']:<30} {entry.warm_start_from or '-':<12} "
        f"{entry.recipe or '-':<8} {'yes' if entry.deliverable else 'no':<12} {gate}"
    )
print("  (* = on this behavior's chain)")

# TRUNK_FROM = "auto" (decision D-A25): now that the chain and RETRAIN_FROM are
# known, pick the trunk run — the run under LOG_BASE/<species>/<algo> whose
# certified ancestors cover the most of this chain root-first under the §4.2
# rule, the greatest run directory name on a tie (the newest timestamp id) —
# and print the choice, what it rests on, every run
# refused and why, and any older-interface parent the command-line widen tool
# could widen. The chain loop then reuses from TRUNK_DIR exactly as for a
# hand-picked run.
if globals().get("AUTO_TRUNK", False):
    from environments.shared.ancestors import select_trunk

    TRUNK_SELECTION = select_trunk(
        LOG_BASE / SPECIES / ALGORITHM.lower(),
        species=SPECIES,
        chain=CHAIN,
        stage_configs=STAGE_CONFIGS,
        plant_identity=PLANT_IDENTITY,
        exclude=(RUN_DIR,),
        retrain_from=RETRAIN_NODE,
        algorithm=ALGORITHM,
    )
    TRUNK_DIR = TRUNK_SELECTION.run_dir
    print("\n" + TRUNK_SELECTION.describe())

# What RUN_DIR already holds constrains this session, and nothing has been trained
# yet. A run whose result bundle is complete is immutable, so a session that would
# judge or train a node into it is refused and pointed at a fresh RUN_ID trunked
# from it, before anything is written (on a complete run the storage cell writes
# nothing; docs/RESULT_BUNDLES.md); the chain loop refuses, before its first write,
# a node this cannot predict (one held only as an ancestors/ record that the trunk
# no longer certifies). A root widened into this run on the command line
# is judged here before a trunk may stand in for it (decision D-C13); that refusal
# comes before anything is trained or judged, though the storage cell has already
# minted this run's provenance.json. Skipped when the storage cell has not run.
if globals().get("RUN_DIR") is not None:
    from environments.shared.result_bundle import (
        refuse_complete_run_session,
        refuse_trunk_over_unjudged_widened_root,
    )

    refuse_complete_run_session(
        RUN_DIR, species=SPECIES, chain=CHAIN, target=TARGET_NODE, retrain_from=RETRAIN_NODE, trunk_dir=TRUNK_DIR
    )
    refuse_trunk_over_unjudged_widened_root(
        RUN_DIR, species=SPECIES, chain=CHAIN, target=TARGET_NODE, retrain_from=RETRAIN_NODE, trunk_dir=TRUNK_DIR
    )

## 4. Preflights

Two checks before anything trains: an SB3 archive loads in this runtime, and the zero-action baseline measures the do-nothing floor against each species' stage-1 gate.

In [ ]:
# SB3 archive load preflight (KNOWN_ISSUES "Training / RL": SB3 archives are bound to the interpreter that saved them)
# An SB3 archive's schedule members (learning_rate / lr_schedule / clip_range) are cloudpickled closures that
# carry the bytecode of the Python that SAVED the archive, and a bare PPO.load executes them while it rebuilds
# the optimizer: under another Python minor version the kernel dies with no traceback. Colab's image moved from
# Python 3.12.13 (the August 2026 runs) to 3.13.15 (already the image of the 2026-09-14/15 runs), and the first
# sessions that loaded an archive saved under the older image died exactly there, inside the widen tool's
# self-verification. Every load in this repository goes through policy_loading.load_sb3_model, which supplies
# those members instead of unpickling them; this cell proves that path on a REAL archive before anything is
# trained: the trunk run's root handoff when there is one, else a throwaway model saved by this very runtime.
# The print is flushed before the load so a kernel death here is attributable to the load. A failure raises
# and halts Run all before any training (the infrastructure cell and its disconnect helper come later).
import tempfile
from pathlib import Path

from environments.shared.curriculum.checkpoints import select_handoff_checkpoint
from environments.shared.policy_loading import inspect_sb3_archive, load_sb3_model
from environments.shared.stage_manifest import stage_dir_candidates


def _preflight_root_handoff(run_dir):
    """The chain root's handoff archive under *run_dir* (either stage-directory naming), or None."""
    for name in stage_dir_candidates(SPECIES, CHAIN[0].reference):
        handoff = select_handoff_checkpoint(Path(run_dir) / name / "models")
        if handoff is not None:
            return Path(handoff[1] + ".zip")
    return None

_preflight_archive = None
_preflight_source = "a throwaway model saved by this runtime"
if TRUNK_DIR is not None:
    _preflight_archive = _preflight_root_handoff(TRUNK_DIR)
    if _preflight_archive is not None:
        _preflight_source = f"the trunk run's root handoff {_preflight_archive}"
with tempfile.TemporaryDirectory(prefix="sb3_load_preflight_") as _preflight_tmp:
    if _preflight_archive is None:
        from stable_baselines3 import PPO

        class _PreflightEnv(gym.Env):
            """Tiny 2-obs/1-action env; exists only to construct a minimal PPO."""

            observation_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
            action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

            def reset(self, *, seed=None, options=None):
                super().reset(seed=seed)
                return np.zeros(2, dtype=np.float32), {}

            def step(self, action):
                return np.zeros(2, dtype=np.float32), 0.0, True, False, {}

        _preflight_archive = Path(_preflight_tmp) / "preflight_ppo.zip"
        PPO("MlpPolicy", _PreflightEnv(), n_steps=32, batch_size=32, policy_kwargs={"net_arch": [8]}, device="cpu").save(
            str(_preflight_archive)
        )
    _preflight_inspection = inspect_sb3_archive(_preflight_archive)
    print(
        f"SB3 archive load preflight: loading {_preflight_source} (saved by Python "
        f"{_preflight_inspection.saved_python_text}; this runtime is Python {sys.version_info[0]}.{sys.version_info[1]}; "
        f"bytecode members: {', '.join(sorted(_preflight_inspection.bytecode_members)) or 'none'}) ... "
        "a kernel death HERE means this image cannot load SB3 archives",
        flush=True,
    )
    _preflight_model = load_sb3_model(_preflight_archive, device="cpu")
    del _preflight_model
print("SB3 archive load preflight passed: archives load back on this runtime through load_sb3_model.")

### Zero-action baseline

The floor that decides whether stage 1 learned anything is the **do-nothing** policy.

Actions are residuals around each species' home keyframe (`home-keyframe-residual/v1`), so
`action = 0` commands the nominal stance exactly. On a passively stable plant that is a real
policy, and on a balance stage it can be a strong one — strong enough that Tyrannosaurus Rex runs
`20260723_204941` and `20260724_140441` both scored *below* it and advanced anyway, because
the gate at the time was `min_avg_reward = 100.0`.

Two numbers come out, and they answer different questions:

| number | meaning |
|---|---|
| **reward** | unconditional mean, including episodes the statue falls out of. A gate below this is cleared by doing nothing. |
| **reward standing** | conditioned on reaching the horizon. A gate between the two is cleared by a policy that has learned only "do not fall". |

`configs/trex/stance.toml` asks for exactly this check: *"this is a measured constant
and will go stale if the plant or the reward weights change — re-run the baseline script and
update it, or better, wire that script in as an automatic pre-stage check."* This cell is that
check. Results are written to Drive so the calibration travels with the run.

By default this measures only the species this notebook is training (~1 min) and writes the
record to that species' own log directory:

```
<LOG_BASE>/<species>/zero_action_baselines/<timestamp>.json   # the numbers, machine-readable
<LOG_BASE>/<species>/zero_action_baselines/<timestamp>.txt    # the table printed below
<RUN_DIR>/zero_action_baseline.json                           # copy, so the run carries its calibration
```

The JSON records the plant identity and the stage-1 env kwargs alongside the numbers, because
those are what make the constant go stale. Widen `BASELINE_SPECIES` to all four (~4 min) to
find out whether a weak gate is species-specific or a shared-template problem — each species'
record still lands in its own directory.

---

To run the same thing from a **Colab terminal** instead of the notebook:

```bash
cd /content
[ -d mesozoic-labs ] || git clone https://github.com/kuds/mesozoic-labs.git
cd mesozoic-labs

# Debian's patched setuptools breaks legacy setup.py sdists; upgrade it first.
pip install -q -U setuptools
pip install -q -e .          # core deps only — the baseline needs no torch/SB3

SPECIES=trex   # or velociraptor / brachiosaurus / dibothrosuchus
OUT=/content/drive/MyDrive/mesozoic-labs/logs/$SPECIES/zero_action_baselines
[ -d /content/drive/MyDrive ] || echo "WARNING: Drive not mounted — saving locally only"
mkdir -p "$OUT"

python environments/shared/scripts/zero_action_baseline.py "$SPECIES" \
    --episodes 40 --seed 3042 2>&1 | tee "$OUT/$(date +%Y%m%d_%H%M%S)_terminal.txt"
```

Drive is only visible to the terminal once the notebook has mounted it (the storage cell of section 3).

Pass several species names to measure them in one go — the transcript then covers all of
them, so `tee` it wherever makes sense rather than into one species' directory. Add
`--sweep-noise` to sweep `reset_noise_scale` instead of measuring a single point; that is
how the stage-1 reset noise gets calibrated.

In [ ]:
# ============================================================
# Pre-flight: zero-action baseline (stage-1 gate calibration)
# ============================================================
# Scores the do-nothing policy on stage 1 and checks each species'
# min_avg_reward gate against it.  These are measured constants: they go stale
# whenever the plant or the stage-1 reward weights change, so the result is
# saved to Drive with the plant identity and the env kwargs that produced it.

from environments.shared.constants import PUBLICATION_SEED_START
from environments.shared.scripts import zero_action_baseline

# Just the species this notebook is training (~1 min).  To find out whether a weak
# gate is species-specific or a shared-template problem, widen this to all four
# (~4 min) -- each species' record still lands in its own log directory:
#   from environments.shared.species_names import species_display_names
#   BASELINE_SPECIES = list(species_display_names())
BASELINE_SPECIES = [SPECIES]
BASELINE_STAGE = 1
BASELINE_EPISODES = 40  # matches the figures recorded in the stage-1 configs (configs/*/, manifest-named)
BASELINE_SEED = PUBLICATION_SEED_START  # the publication_evaluation seed role

# One record per species under <LOG_BASE>/<species>/zero_action_baselines/, and a copy
# in RUN_DIR (what curriculum.baseline_watch reads) unless that run's bundle is complete.
zero_action_baseline.preflight(
    BASELINE_SPECIES,
    stage=BASELINE_STAGE,
    episodes=BASELINE_EPISODES,
    seed=BASELINE_SEED,
    species=SPECIES,
    log_base=globals().get("LOG_BASE"),
    run_dir=globals().get("RUN_DIR"),
)

## Helpers (nothing to set)

Definitions only: `train_stage` and the chain state the loop fills, then the plotting wrappers the chain loop and the report use.

In [ ]:
from environments.shared import train_base
from environments.shared.config import read_stage_duration
from environments.shared.notebook_runtime import disconnect_runtime, display_stage_videos, halt
from environments.shared.replication import discover_replicates_for_run
from environments.shared.reporting import evaluate_stage_checkpoints, generate_stage_artifacts
from environments.shared.reporting import (
    save_result_bundle as _lib_save_result_bundle,
)
from environments.shared.reporting import (
    write_training_summary as _lib_write_training_summary,
)

# Chain state (BEHAVIOR_RECIPES_PLAN §4.7). The chain loop fills these as it
# walks BEHAVIOR's chain; the tail cells read them instead of hand-threaded
# per-stage variables.
#   completed_stages: (stage reference, stage_dir) for every node trained or
#     judged in THIS run, in completion order — what the curves cells plot.
#   NODE_RESULTS: stage id -> stage_results dict for every node this run
#     holds results for. A cross-run reused ancestor has none: it writes no
#     CSV row here — its record lives under ancestors/.
#   NODE_HANDOFF: stage id -> the certified handoff its children load from:
#     {"model": <stem>, "vecnorm": <path>, "stage_dir": Path, "run_dir": Path,
#      "run_id": <str | None>, "model_sha256": str, "normalization_sha256": str, "reused": bool}.
#     run_id is the ancestor's run for a CROSS-run reuse (the child records
#     it as parent_run_id) and None for a node from this run; model_sha256
#     is the digest the next node's reuse rule chains on (D-A17). Certified
#     nodes only — a node that failed its gate never enters it.
completed_stages = []
NODE_RESULTS = {}
NODE_HANDOFF = {}


def chain_results():
    """NODE_RESULTS' values in manifest order — the list every summary and bundle write passes."""
    return [NODE_RESULTS[entry.id] for entry in MANIFEST.stages if entry.id in NODE_RESULTS]


def train_stage(
    stage,
    timesteps,
    *,
    load_path=None,
    run_dir=None,
    vecnorm_path=None,
    task_load_mode,
    parent_run_id=None,
    label=None,
    eval_freq=50_000,
    save_freq=100_000,
):
    """Train one node of the behavior chain with ``train_base.train``, the CLI's trainer, then evaluate it.

    ``task_load_mode`` is REQUIRED and never inferred. The chain loop passes
    ``"initialize_next_stage"`` when the node loads its parent's handoff
    (its declared edge, recorded as lineage) and ``"resume_same_stage"`` for
    a root that loads nothing; the RESUME cell passes ``"resume_same_stage"``
    for a load of this node's OWN periodic checkpoint. A checkpoint and its
    VecNormalize sidecar always travel together. ``parent_run_id`` names
    the run a reused certified ancestor came from (``None`` for a parent
    trained in this run); ``label`` is the free-text run label recorded
    beside the hyperparameter digest (decision D-A21). What ``train`` refuses
    and records, and the two switches passed here, are in its docstring; the
    gate is judged afterwards by ``generate_stage_artifacts`` and enforced by
    the chain loop.

    Returns (model, best_model_path, final_model_path, stage_dir, vecnorm_save_path, stage_results).
    """
    NODE = MANIFEST.resolve(stage)
    if task_load_mode not in ("initialize_next_stage", "resume_same_stage"):
        raise ValueError(
            "task_load_mode must be declared as 'initialize_next_stage' (the parent's handoff) or "
            f"'resume_same_stage' (this node's own checkpoint, or nothing), not {task_load_mode!r}"
        )
    if bool(load_path) != bool(vecnorm_path):
        raise ValueError(
            "load_path and vecnorm_path must be given together: a checkpoint is only ever loaded under "
            "the VecNormalize statistics it was trained with"
        )
    if task_load_mode == "initialize_next_stage" and not load_path:
        raise ValueError(f"{NODE.id!r}: 'initialize_next_stage' declares a load from the parent's handoff, but no load_path was given")
    if task_load_mode == "initialize_next_stage" and NODE.warm_start_from is None:
        raise ValueError(
            f"{NODE.id!r} is a root of the {SPECIES} manifest (no warm_start_from), so it cannot enter from a "
            "parent under 'initialize_next_stage'; a root trains from scratch or resumes its own checkpoint"
        )
    if vecnorm_path and not Path(vecnorm_path).is_file():
        # Refused here, before train() writes the stage config, so a corrected re-run is not an occupied directory.
        raise FileNotFoundError(
            f"VecNormalize sidecar not found: {vecnorm_path}. Refusing to train the loaded checkpoint under "
            "fresh normalization statistics."
        )
    # Directory names sort in curriculum order and carry the id (01_stance,
    # 02_recovery, ...); files inside keep their stage_label prefixes.
    stage_dir = Path(run_dir if run_dir is not None else repo_root / "logs") / stage_dirname(SPECIES, stage)
    config = STAGE_CONFIGS[stage]
    print(f"{'=' * 60}")
    print(f"Node {NODE.id} (stage {stage}): {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    if vecnorm_path:
        print(f"VecNormalize loaded from: {vecnorm_path}")
    print(f"{'=' * 60}")

    model = train_base.train(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        timesteps,
        n_envs=N_ENVS,
        seed=SEED,
        load_path=load_path,
        vecnorm_path=vecnorm_path,
        eval_freq=eval_freq,
        save_freq=save_freq,
        verbose=VERBOSE,
        algorithm=ALGORITHM.lower(),
        output_dir=str(stage_dir),
        task_load_mode=task_load_mode,
        parent_run_id=parent_run_id,
        label=label,
        report_metrics=False,
        save_on_interrupt=False,
    )
    final_path = stage_dir / "models" / f"{stage_label(stage)}_final"
    print(f"\nFinal model saved to: {final_path}.zip")

    _model, _handoff_stem, _final_model_path, _handoff_vecnorm, _stage_results = evaluate_stage_checkpoints(
        SPECIES_CFG,
        config,
        stage,
        ALGORITHM,
        stage_dir,
        final_path=final_path,
        final_vecnorm_path=f"{final_path}_vecnorm.pkl",
        timesteps=int(model.num_timesteps),
        duration_seconds=read_stage_duration(stage_dir),
        plant_identity=PLANT_IDENTITY,
        evaluation_seed=EVALUATION_SEED,
        model=model,
    )
    return _model, _handoff_stem, _final_model_path, stage_dir, _handoff_vecnorm, _stage_results


def write_training_summary(run_dir, stage_results_list, species=None):
    """Write a training summary text file to the run directory."""
    if species is None:
        species = SPECIES
    path = _lib_write_training_summary(
        run_dir,
        stage_results_list,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        n_envs=N_ENVS,
        quick_test=QUICK_TEST,
    )
    summary_text = path.read_text()
    print(f"\nTraining summary saved to: {path}")
    print(summary_text)


def save_run_bundle(stage_results_list, run_dir=None, species=SPECIES):
    """Regenerate the canonical, Drive-portable bundle for the nodes this run holds results for.

    The bundle is `complete` only when BEHAVIOR's node (the target
    deliverable) is present and certified and every deliverable present is
    certified; a chain stopped above the target leaves it `partial`, which
    still publishes every certified deliverable (result schema v4).

    Seed replication (BEHAVIOR_RECIPES_PLAN §4.5, decision D-B16): the sibling
    runs under this species/algorithm log directory that certified the same
    recipe (same task, plant, gate and hyperparameter digests) on another seed
    are recorded as each deliverable's replicates, and a deliverable with fewer
    certifying runs than its config's `certification_seeds` is labelled
    provisional. Re-run this cell once a replicate finishes to count it: a
    partial bundle is rebuilt, and a complete one regenerates its derived
    artifacts when the replication record is its only change (everything
    else in it stays immutable).
    """
    if run_dir is None:
        run_dir = RUN_DIR
    paths = _lib_save_result_bundle(
        stage_results_list,
        target_deliverable=TARGET_NODE.key,
        stage_configs=STAGE_CONFIGS,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        run_dir=run_dir,
        backend="stable-baselines3",
        hardware=HARDWARE_LABEL,
        parallel_envs=N_ENVS,
        evaluation_episodes=30,
        evaluation_seeds=[CHECKPOINT_SELECTION_SEED, EVALUATION_SEED],
        seed_roles={
            "training": SEED,
            "checkpoint_selection_evaluation": CHECKPOINT_SELECTION_SEED,
            "publication_evaluation": EVALUATION_SEED,
            "certification_panel": PUBLICATION_SEED_START,
        },
        plant_identity=PLANT_IDENTITY.to_dict(),
        run_id=_ACTIVE_RUN_ID,
        repository_root=repo_root,
        replicates=discover_replicates_for_run(run_dir, species=species, plant_identity=PLANT_IDENTITY),
    )
    print("\nResult bundle updated:")
    for name, path in paths.items():
        print(f"  {name}: {path}")
    return paths



print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")
print("DiagnosticsCallback enabled: per-component rewards, obs/action stats,")
print("VecNormalize tracking, and termination reasons")
print("will log to TensorBoard.")

In [ ]:
from environments.shared.visualization import (
    plot_diagnostics_graphs as _lib_plot_diagnostics_graphs,
)
from environments.shared.visualization import (
    plot_training_curves as _lib_plot_training_curves,
)


def plot_training_curves(stage_dirs, stage_configs, algo_name, save_path=None):
    """Plot evaluation reward, episode length, tilt angle, and forward velocity curves.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global and calls ``plt.show()`` for inline display.
    """
    _lib_plot_training_curves(
        stage_dirs,
        stage_configs,
        species=SPECIES,
        algorithm=algo_name,
        save_path=save_path,
    )
    if save_path is not None:
        print(f"Training curves saved to: {save_path}")
    plt.show()


def plot_diagnostics_graphs(stage_dirs, stage_configs, algo_name, save_dir=None, _show=True):
    """Create diagnostic figures for locomotion health and behavioral metrics.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global.  When ``_show`` is True (default), figures
    are displayed inline; otherwise they are closed after saving.
    """
    fig1, fig2 = _lib_plot_diagnostics_graphs(
        stage_dirs,
        stage_configs,
        species=SPECIES,
        algorithm=algo_name,
        save_dir=save_dir,
        show=_show,
    )
    if _show:
        plt.show()


print("Visualization functions ready (shared library).")

## 5. Resume an Interrupted Node (Colab runtime cap)

Colab reclaims the runtime at its session cap (~24 h), and a long node does not
survive it: run `20260821_142144` lost locomotion at 5,488,640 of its 8M-step
budget — no `stage2_final`, no summary, no figures. What does survive are the
periodic checkpoints (`CheckpointCallback`, every 100k steps; that run trained
at the former 500k cadence, so its newest was `stage2_5000000_steps.zip` beside
its matched `stage2_vecnormalize_5000000_steps.pkl` — a ~488k-step / ~2.2 h
loss window the denser cadence shrinks to at most ~30 min, and retention keeps
5 step-points so storage is unchanged). This cell resumes an interrupted stage
from its newest **intact** periodic checkpoint, turning the cap from a
run-killer into a checkpoint boundary.

In a fresh runtime:

1. Set `RUN_ID` in the configuration cell (section 2) to the interrupted run's id
   (e.g. `"20260821_142144"`) so `RUN_DIR` points at the existing run directory
   instead of minting a new one, and set `SPECIES`, `BEHAVIOR` and `SEED` as the
   interrupted session had them. Set `RETRAIN_FROM = ""`: the chain loop trains a
   node it covers instead of judging it, so this cell refuses to resume one. When the
   node is an ancestor rather than `BEHAVIOR`'s target and a trunk run also certifies
   it, set `TRUNK_FROM = ""` too: the loop would reuse the trunk's copy before its
   JUDGE branch reaches the resumed one.
2. Set `RESUME_STAGE` below to the interrupted node — its id (`"locomotion"`,
   `"recovery"`) or its stage number (`2`: the N of its `stageN_*` checkpoint files,
   not the resolve table's `#` or the `NN_` directory prefix); the chain loop's
   "interrupted node" message names it. A node off `BEHAVIOR`'s chain is refused: the
   chain loop would never judge it.
3. Run all. The storage cell's `Run directory:` line reads "re-entering run"; this cell
   trains the rest of the budget and evaluates it, but writes no gate verdict, summary
   or bundle; the chain loop (section 6) then judges the node with its JUDGE branch,
   records it and continues down the chain.

A node with nothing left to train is never retrained: when it already holds a
`gate_verdict.json` (the chain loop reuses or refuses it) or an intact final checkpoint
pair (`<stage_label>_final.zip` and its `_vecnorm.pkl`, checked like a periodic pair:
training finished, perhaps stopped early by a callback, and the loop's JUDGE branch
judges it), this cell prints so and trains nothing, so a `RESUME_STAGE` left set does not stop a later Run all. Training such
a node further is a new attempt, in a fresh `RUN_ID`.

Pointing `RUN_ID` at an old run is for resuming *that run's* interrupted node only, and
only while its bundle is not `complete`: a complete bundle is immutable, and this cell
refuses to train into one. Certified nodes of an earlier run come in through
`TRUNK_FROM` in a **new** `RUN_ID` — never by pointing `RUN_ID` at the old run (the
storage cell refuses to re-mint a run directory whose `provenance.json` records another
plant identity or seed, and the chain loop refuses a verdict its reuse rule rejects
with "mint a fresh RUN_ID").

A certified checkpoint from a run trained behind this checkout's policy interface is
widened on the command line into a new run, one this notebook has not opened yet
(`python -m environments.shared.scripts.widen_checkpoint ... --to-stage-dir
<LOG_BASE>/<species>/<algo>/<new run id>/<root stage directory>`, where `<new run id>`
is a new timestamp id, `YYYYMMDD_HHMMSS` like the ones the storage cell mints, with
`--max-revision-gap N` for a parent more than one revision behind, decision D-C17, and
`--label "<RUN_LABEL>"` when this session sets `RUN_LABEL`; on Colab, the tool's module
docstring lists the three steps: section 1, a scratch cell that mounts Drive, never the
storage cell, then the tool from `/content/mesozoic-labs`). This notebook
then judges the widened root with `RUN_ID` set to that run, `SEED` set to the parent
run's seed and `TRUNK_FROM = ""` (decisions D-C13, D-C14, D-D14: the storage cell
refuses any other `SEED` before it writes anything, and the resolve cell refuses a
trunk until the widened root holds a verdict). `N_ENVS` in that session describes the
nodes trained there; the widened root's run block keeps the parent's `n_envs`.

The cell globs the stage's `models/` directory for
`<stage_label>_<steps>_steps.zip` and walks the step-points newest first,
validating each candidate pair before trusting it: the matched
`<stage_label>_vecnormalize_<steps>_steps.pkl` sidecar must exist, the
checkpoint zip must pass an integrity check, and the sidecar must unpickle. A
truncated or orphaned newest pair — exactly what an ungraceful runtime reclaim
mid-write leaves behind — is skipped with a warning and the next older
step-point is used instead, so a corrupt newest pair no longer needs deleting
by hand. The first intact pair wins, and the cell trains for the remaining
budget (stage timesteps minus the chosen checkpoint's steps). The load is
validated as `resume_same_stage`, so a config edited between sessions fails
closed on the task fingerprint instead of silently resuming across a task
change. A candidate missing its VecNormalize sidecar is never resumed:
training a loaded policy under fresh normalization statistics is the
silent-collapse failure mode of review F3. Only when **no** step-point
survives validation does the cell raise, listing every skipped file and why.

Resume-from-periodic-checkpoint requires the sidecar fallback fix in
`train_base` (this branch, review F3) — before it, checkpoint loading probed
only the curated `<base>_vecnorm.pkl` name, warned, and trained under fresh
statistics silently. The chain loop (section 6) then finds the node's final
checkpoint without a gate verdict, evaluates it, judges the gate and records the
verdict, then continues down the chain. The
recorded duration is the resumed session's: a session stopped before its
final save records none.

In [ ]:
# ===== RESUME AN INTERRUPTED STAGE — opt-in escape hatch =====
RESUME_STAGE = None  # None = skip; the node's id ("locomotion", "recovery") or its stageN number (2) to resume

if RESUME_STAGE is not None:
    import pickle
    import re
    import zipfile

    from environments.shared.result_bundle import read_gate_verdict, refuse_write_into_complete_run
    from environments.shared.stage_manifest import stage_dirname, stage_label

    # An id or a number, as in the manual cell: STAGE_CONFIGS, the checkpoint
    # names and the chain loop use the stage's reference, its number when it has one.
    entry_res = MANIFEST.resolve(RESUME_STAGE)
    if entry_res not in CHAIN:
        raise RuntimeError(
            f"RESUME_STAGE={RESUME_STAGE!r} is {entry_res.id!r}, which is not on the chain of behavior "
            f"{BEHAVIOR!r} ({[node.id for node in CHAIN]}): the chain loop would never judge it. Set SPECIES, "
            "BEHAVIOR and SEED as the interrupted session had them and re-run sections 2-3."
        )
    # RETRAIN_FROM trains every node it covers from its parent instead of judging it, and D-A20
    # refuses that node's occupied directory: a node resumed under it could never be judged.
    retrain_res = globals().get("RETRAIN_NODE")
    if retrain_res is not None and (entry_res == retrain_res or retrain_res in MANIFEST.ancestors(entry_res.id)):
        raise RuntimeError(
            f"RESUME_STAGE={RESUME_STAGE!r} is {entry_res.id!r}, which RETRAIN_FROM (it resolved to "
            f"{retrain_res.id!r}) covers: the chain loop would train it again from its parent instead of judging "
            'the resumed node, and refuses its occupied directory. Set RETRAIN_FROM = "" and re-run sections 2-3.'
        )
    stage_res = entry_res.reference
    cfg_res = STAGE_CONFIGS[stage_res]
    # Same budget derivation as the chain loop: the remaining
    # budget must be measured against the budget the interrupted run used.
    budget_res = 50_000 if QUICK_TEST else cfg_res["curriculum_kwargs"]["timesteps"]
    label_res = stage_label(stage_res)
    stage_dir_res = RUN_DIR / stage_dirname(SPECIES, stage_res)
    model_dir_res = stage_dir_res / "models"

    def pair_problem_res(zip_path, vecnorm_path):
        """None when a checkpoint zip and its VecNormalize sidecar load, else why not: a truncated or
        orphaned pair is exactly what an ungraceful runtime reclaim mid-write leaves behind. A pair missing
        its sidecar is never trusted: a loaded policy under fresh statistics collapses silently (review F3)."""
        if not vecnorm_path.exists():
            return f"missing matched VecNormalize sidecar {vecnorm_path.name}"
        try:
            with zipfile.ZipFile(zip_path) as zf_res:
                bad_member_res = zf_res.testzip()
                names_res = zf_res.namelist()
            if bad_member_res is not None:
                raise zipfile.BadZipFile(f"corrupt archive member {bad_member_res!r}")
            # A truncated SB3 checkpoint can still open: zipfile locks onto a nested torch archive's
            # end-of-directory record, so testzip alone passes. Require SB3's own members in the OUTER archive.
            if "data" not in names_res or not any(n.endswith("policy.pth") for n in names_res):
                raise zipfile.BadZipFile(
                    f"outer archive lacks SB3 members (truncated checkpoint; found {sorted(names_res)[:5]}...)"
                )
        except Exception as exc:
            return f"bad/truncated checkpoint zip {zip_path.name} ({exc})"
        try:
            with open(vecnorm_path, "rb") as f:
                pickle.load(f)
        except Exception as exc:
            return f"VecNormalize sidecar {vecnorm_path.name} does not unpickle ({exc})"
        return None

    # A node that holds a verdict or an intact final pair has nothing to resume, whatever its
    # periodic checkpoints say (an early stop, or an N_ENVS that does not divide the
    # checkpoint cadence, leaves the newest one short of the budget): the chain loop
    # reuses or refuses a judged node and judges a finished one with its JUDGE branch. A
    # final pair a reclaim cut short during the final save is resumed over instead.
    verdict_res = read_gate_verdict(stage_dir_res)
    final_pair_res = [model_dir_res / f"{label_res}_final{suffix}" for suffix in (".zip", "_vecnorm.pkl")]
    final_problem_res = pair_problem_res(*final_pair_res) if final_pair_res[0].exists() else None
    if verdict_res is not None or (final_pair_res[0].exists() and final_problem_res is None):
        held_res = "a gate verdict" if verdict_res is not None else f"its final pair {final_pair_res[0].name}"
        print(
            f"Nothing to resume: {entry_res.id!r} already holds {held_res} in {stage_dir_res}; a finished node "
            "is never retrained. The chain loop (section 6) reuses or refuses a judged node and judges a finished "
            "one with its JUDGE branch; training it further is a new attempt, in a fresh RUN_ID."
        )
    else:
        if final_problem_res is not None:
            print(
                f"WARNING: the final pair of {entry_res.id!r} is incomplete ({final_problem_res}): the runtime "
                "stopped during the final save, so the node resumes from its newest intact periodic pair, and "
                "that resume's final save replaces the broken pair."
            )
        print(f"Resuming {entry_res.id!r} (stage {stage_res!r}) from {model_dir_res}")

        # SB3's CheckpointCallback names periodic checkpoints
        # {prefix}_{steps}_steps.zip and (save_vecnormalize=True in
        # _build_core_callbacks) writes a matched
        # {prefix}_vecnormalize_{steps}_steps.pkl sidecar.
        ckpts_res = []
        for p in sorted(model_dir_res.glob(f"{label_res}_*_steps.zip")):
            m = re.fullmatch(re.escape(label_res) + r"_(\d+)_steps", p.stem)
            if m:
                ckpts_res.append((int(m.group(1)), p))
        if not ckpts_res:
            raise RuntimeError(
                f"No periodic checkpoint {label_res}_*_steps.zip in {model_dir_res} — "
                "check that RUN_ID in the configuration cell names the interrupted run, and run the storage "
                f"cell (section 3) after changing it; and that RESUME_STAGE names its node (it resolved to "
                f"{entry_res.id!r}: a number is the N of the stageN_* checkpoint names, not the resolve table's # "
                "or the NN_ prefix)."
            )
        # Newest-first candidate walk with integrity validation: the newest pair
        # is exactly the file an ungraceful runtime reclaim may have left
        # truncated or orphaned mid-write, so a bad newest candidate falls back
        # to the next older step-point instead of killing the resume.
        steps_res = ckpt_res = vecnorm_ckpt_res = None
        skipped_res = []
        for cand_steps, cand_ckpt in sorted(ckpts_res, key=lambda sc: sc[0], reverse=True):
            cand_vecnorm = model_dir_res / f"{label_res}_vecnormalize_{cand_steps}_steps.pkl"
            problem_res = pair_problem_res(cand_ckpt, cand_vecnorm)
            if problem_res is not None:
                reason = f"{cand_ckpt.name}: {problem_res}"
                print(f"WARNING: skipping {reason}")
                skipped_res.append(reason)
                continue
            steps_res, ckpt_res, vecnorm_ckpt_res = cand_steps, cand_ckpt, cand_vecnorm
            break
        if ckpt_res is None:
            raise FileNotFoundError(
                f"No intact periodic checkpoint pair {label_res}_<steps>_steps.zip + "
                f"{label_res}_vecnormalize_<steps>_steps.pkl in {model_dir_res}. Skipped: "
                + "; ".join(skipped_res)
                + ". Deleting a corrupt newest pair by hand is no longer needed — this "
                "cell already fell back through every older step-point."
            )
        remaining_res = max(budget_res - steps_res, 0)
        print(f"Newest intact periodic checkpoint: {ckpt_res}")
        print(f"Matched VecNormalize:              {vecnorm_ckpt_res}")
        if skipped_res:
            print(f"({len(skipped_res)} newer candidate(s) skipped as incomplete/corrupt — see warnings above)")
        print(f"Checkpoint steps: {steps_res:,} of {budget_res:,} — remaining budget: {remaining_res:,}")

        if remaining_res == 0:
            # The final save follows the last periodic one; a runtime stopped between the two, or
            # during the final save, leaves a spent budget with no intact final pair, which the
            # chain loop's JUDGE branch needs.
            lost_res = "was never saved" if final_problem_res is None else f"is incomplete ({final_problem_res})"
            raise RuntimeError(
                f"{ckpt_res.name} already covers the {budget_res:,}-step budget of {entry_res.id!r}, but its final "
                f"pair {final_pair_res[0].name} {lost_res} (the runtime stopped before the final save completed), so "
                "the chain loop's JUDGE branch cannot judge it and there is nothing left to train: train the node "
                "again in a fresh RUN_ID, with TRUNK_FROM naming this run for its certified ancestors."
            )
        else:
            # A complete bundle is immutable: nothing is resumed into it (this refusal writes nothing).
            refuse_write_into_complete_run(RUN_DIR, what=f"Resuming {stage_res!r}")
            model_res, path_res, final_path_res, dir_res, vecnorm_res, results_res = train_stage(
                stage=stage_res,
                timesteps=remaining_res,
                load_path=str(ckpt_res),
                run_dir=RUN_DIR,
                vecnorm_path=str(vecnorm_ckpt_res),
                task_load_mode="resume_same_stage",
                label=RUN_LABEL or None,
            )

## 6. Train the behavior chain

One loop walks `BEHAVIOR`'s chain from the species' stage manifest, root first, and for each node does exactly one of:

1. **Reuse** a certified checkpoint — from this run (a re-run of this cell in the same `RUN_DIR`) or, for an ancestor, from the trunk run (`TRUNK_FROM`: a pinned run, or under `"auto"` the sibling run selected in the resolve cell) — when the reuse rule holds: a passed gate verdict, the same task digest and plant, and a checkpoint that descends from the parent resolved here. A cross-run ancestor is recorded under `ancestors/` (small JSON records, never the checkpoint pair) and its handoff pair is loaded from the run that certified it; the target node is never reused across runs.
2. **Judge** a node that was trained but never gated (its final checkpoint exists, `gate_verdict.json` does not — the RESUME cell above finished its budget, or the command-line widen tool wrote a widened root into this run): evaluate its checkpoints and judge the gate.
3. **Train** it otherwise, warm-started from its parent's handoff checkpoint and VecNormalize sidecar along the declared edge (`initialize_next_stage`); a root loads nothing. For a frozen-null gate kind (recovery) the gate's thresholds and null panels freeze **before** training and the policy panel rolls after, on the same frozen seeds.

Every trained or judged node writes its artifacts, the training summary and the run bundle **before** its verdict is enforced, so a failed gate still publishes every certified deliverable above it; the runtime is then released and the loop raises. `RETRAIN_FROM` trains the named node and everything below it instead of reusing them; a run directory that already records a node is refused, so a new variant is a new `RUN_ID`. A run whose bundle is already `complete` is immutable: the resolve cell refuses a session that would train or judge a node into it, and this loop refuses any such write it could not predict before the write, so a later node goes into a fresh `RUN_ID` whose `TRUNK_FROM` names that run. A widened root this run holds without a verdict is judged here, never replaced by a trunk's copy: the resolve cell refuses a trunk until it holds one (decision D-C13). Under `BEHAVIOR = "stand"` on a species with a recovery stage, the recovery verdict is enforced like any other node's.

In [ ]:
# ===== BEHAVIOR CHAIN LOOP =====
# Walks BEHAVIOR's chain root-first (BEHAVIOR_RECIPES_PLAN §4.7, decision D5)
# and, per node, does exactly one of:
#   (1) REUSE — a certified checkpoint already exists in RUN_DIR (a same-run
#       re-run) or, for an ancestor, in TRUNK_FROM's run (§4.2: passed verdict,
#       same task digest and plant, chained by digest onto the parent resolved
#       here — D-A17). TRUNK_FROM = "auto" (D-A25) selects TRUNK_DIR in the
#       resolve cell: the sibling run covering the most of the chain. A cross-run
#       ancestor is recorded under ancestors/ and its handoff is loaded from the
#       run that certified it (A10); the target node is never reused across runs (D-A18).
#   (3) JUDGE — trained (final checkpoint + sidecar exist; the RESUME cell spent
#       its budget, or the command-line widen tool wrote a widened root) but never
#       judged: evaluate, then gate it like a trained node.
#   (2) TRAIN — otherwise, from the parent's handoff along the declared edge
#       ("initialize_next_stage"); a root loads nothing.
# A frozen-null gate kind (recovery) freezes its resolution BEFORE the node
# trains and rolls the policy panel after. Every trained or judged node writes
# its artifacts and the run bundle BEFORE the verdict is enforced — a failed
# gate still publishes every certified deliverable above it — then releases
# the runtime and raises. RETRAIN_FROM (D-A19) removes reuse for the named node
# and its descendants; a recorded stage directory refuses to be overwritten
# (D-A20), so every variant is a new run.
import json

from environments.shared.ancestors import AncestorReuseError, find_certified_ancestor, record_ancestor
from environments.shared.config import hyperparameter_diff
from environments.shared.curriculum import FROZEN_NULL_GATE_KINDS
from environments.shared.harnesses.freeze_recovery_gate import (
    freeze_recovery_gate,
    roll_policy_panel,
    validate_recovery_resolution,
)
from environments.shared.recovery_evaluation import write_recovery_evidence
from environments.shared.result_bundle import (
    ANCESTOR_RECORD_NAME,
    ANCESTORS_DIRNAME,
    read_gate_verdict,
    refuse_write_into_complete_run,
    sha256_file,
)
from environments.shared.task_fingerprint import (
    derive_stage_task_fingerprint,  # noqa: F811 - this cell also runs standalone
)

print(f"Behavior {BEHAVIOR!r} -> {TARGET_NODE.id}; chain: {' -> '.join(node.id for node in CHAIN)}")
if globals().get("AUTO_TRUNK", False):
    # D-A25: under TRUNK_FROM = "auto" the trunk is the resolve cell's selection for THIS
    # log directory — never a TRUNK_DIR the storage cell reset on a rerun (it starts at
    # None and only a pinned TRUNK_FROM fills it there).
    _selection = globals().get("TRUNK_SELECTION")
    if _selection is None or _selection.log_dir != LOG_BASE / SPECIES / ALGORITHM.lower():
        raise RuntimeError(
            "TRUNK_FROM = 'auto' but the resolve cell has not selected a trunk for this log directory in this "
            "kernel; re-run the resolve cell (section 3) before the chain loop"
        )
    TRUNK_DIR = _selection.run_dir
for NODE in CHAIN:
    stage = NODE.reference
    config = STAGE_CONFIGS[stage]
    stage_dir = Path(RUN_DIR) / stage_dirname(SPECIES, stage)
    gate_kind = config.get("curriculum_kwargs", {}).get("gate_kind")
    budget = 50_000 if QUICK_TEST else config["curriculum_kwargs"]["timesteps"]
    print(f"\n{'#' * 60}\n# Node {NODE.id} (stage {stage}): {config['name']} — gate {gate_kind}\n{'#' * 60}")

    # The parent resolves first: a node is satisfied — by a checkpoint from
    # anywhere — only on top of its declared parent's certified handoff.
    parent = MANIFEST.parent_of(NODE.id)
    if parent is not None and parent.id not in NODE_HANDOFF:
        raise RuntimeError(
            f"{NODE.id!r} warm-starts from {parent.id!r}, which is not certified in this session (NODE_HANDOFF). "
            "Re-run this cell from the top of the chain; a parent that failed its gate stops the chain there."
        )
    parent_handoff = NODE_HANDOFF[parent.id] if parent is not None else None
    task_sha256 = derive_stage_task_fingerprint(
        species=SPECIES,
        stage=stage,
        backend="stable-baselines3",
        env_kwargs=config.get("env_kwargs", {}),
        plant_identity=PLANT_IDENTITY.to_dict(),
    )["task_sha256"]

    # D-A19: RETRAIN_FROM covers the named node and every descendant — no reuse
    # candidates at all, they train here. D-A18: the target is only ever reused
    # from THIS run (an earlier run's target is that run's deliverable);
    # ancestors also look in TRUNK_DIR. RUN_DIR is tried first; only the
    # trunk candidate may follow ancestor records (D-A23).
    covered = RETRAIN_NODE is not None and (NODE.id == RETRAIN_NODE.id or RETRAIN_NODE in MANIFEST.ancestors(NODE.id))
    if covered:
        candidates = []
        print(f"Not reusing {NODE.id!r}: RETRAIN_FROM={RETRAIN_FROM!r} covers it. Training it here.")
    elif NODE.id == TARGET_NODE.id:
        candidates = [RUN_DIR]
    else:
        candidates = [RUN_DIR] + ([TRUNK_DIR] if TRUNK_DIR is not None else [])

    # (1) REUSE — every refusal is printed with its reason, never silent.
    ancestor = None
    for candidate in candidates:
        try:
            ancestor = find_certified_ancestor(
                candidate,
                species=SPECIES,
                entry=NODE,
                current_task_sha256=task_sha256,
                plant_identity=PLANT_IDENTITY,
                # D-A22 (rule 7): the block this session would judge the node under —
                # the one save_stage_config records as 'curriculum'.
                current_gate_config=config.get("curriculum_kwargs", {}),
                parent_model_sha256=parent_handoff["model_sha256"] if parent_handoff is not None else None,
                # D-A23: a trunk that itself reused a node resolves it through its
                # ancestors/ record to the run that certified it. Never for RUN_DIR:
                # this run's own record marks a cross-run reuse, not a node of ours.
                follow_records=candidate is not RUN_DIR,
            )
            break
        except AncestorReuseError as exc:
            print(f"Not reusing {NODE.id!r} from {candidate}: {exc}")
    if ancestor is not None:
        same_run = candidate == RUN_DIR
        # D-A21: reuse carries the ancestor's recipe. An edit to this node's
        # algorithm block or shaping keys since the ancestor was trained is
        # IGNORED — say so, and say how to train it here. Never a refusal.
        try:
            _recorded = json.loads((ancestor.stage_dir / "stage_config.json").read_text(encoding="utf-8"))
            ignored_edits = hyperparameter_diff(config, ALGORITHM, _recorded)
        except (OSError, ValueError):
            ignored_edits = ["<unreadable stage_config.json>"]
        if ignored_edits:
            print(
                f"WARNING: reusing certified {NODE.id!r} from run {ancestor.run_id} ignores this run's hyperparameter "
                f"edit: {', '.join(ignored_edits)} differ from the ancestor's recorded stage_config.json, and reuse "
                f"trains nothing. Set RETRAIN_FROM = {NODE.id!r} to train {NODE.id!r} here under the edited configuration."
            )
        if same_run:
            # This run's own certified node (a re-run of this cell in the same
            # RUN_DIR): its results re-enter the bundle from the verdict.
            stage_result = ancestor.verdict.get("stage_result")
            if not isinstance(stage_result, dict):
                raise RuntimeError(f"{ancestor.stage_dir / 'gate_verdict.json'} records no stage_result to re-enter the bundle with")
            NODE_RESULTS[NODE.id] = dict(stage_result)
            completed_stages.append((stage, ancestor.stage_dir))
            print(f"Reusing this run's certified {NODE.id!r}: {ancestor.handoff_name} ({ancestor.model_stem})")
        else:
            # A cross-run ancestor is recorded under ancestors/ with its original
            # provenance (small JSON records, never the checkpoint pair); the
            # handoff loads it from the run that certified it (A10). A complete
            # bundle is immutable: a record it does not hold yet is refused.
            if not (Path(RUN_DIR) / ANCESTORS_DIRNAME / NODE.id / ANCESTOR_RECORD_NAME).is_file():
                refuse_write_into_complete_run(RUN_DIR, what=f"Recording {NODE.id!r} from run {ancestor.run_id}")
            record_ancestor(RUN_DIR, ancestor)
            print(f"Reusing certified {NODE.id!r} from run {ancestor.run_id}: {ancestor.handoff_name} ({ancestor.model_stem})")
        NODE_HANDOFF[NODE.id] = {
            "model": ancestor.model_stem,
            "vecnorm": str(ancestor.normalization_path),
            "stage_dir": ancestor.stage_dir,
            "run_dir": ancestor.source_run_dir,
            "run_id": None if same_run else ancestor.run_id,
            "model_sha256": ancestor.model_sha256,
            "normalization_sha256": ancestor.normalization_sha256,
            "reused": True,
        }
        continue

    # What the stage directory already holds decides between JUDGE and TRAIN.
    # An existing verdict the reuse rule refused is never retrained over.
    model_dir = stage_dir / "models"
    final_stem = model_dir / f"{stage_label(stage)}_final"
    final_vecnorm = f"{final_stem}_vecnorm.pkl"
    verdict = read_gate_verdict(stage_dir)
    if verdict is not None and not covered:
        if not verdict["passed"]:
            raise RuntimeError(
                f"{NODE.id!r} already holds a FAILED gate verdict in {stage_dir}: " + "; ".join(verdict["failures"])
                + ". A failed node is never silently retrained: start a new run (a fresh RUN_ID, an id no run uses: "
                'RUN_ID = "" re-enters this run through the memo) for a new attempt.'
            )
        raise RuntimeError(
            f"{NODE.id!r} holds a passed gate verdict in {stage_dir} that the reuse rule refused (the reason is "
            "printed above): its task, plant, gate or parent no longer matches this session. A changed task is a new "
            'run — mint a fresh RUN_ID (an id no run uses: RUN_ID = "" re-enters this run through the memo). A changed '
            "gate is a re-judge, never a retrain: remove that directory's gate_verdict.json so this cell's JUDGE branch "
            "re-judges its checkpoints under this session's gate, or run scripts/backfill_gate_verdict.py --force "
            "[--gate current] on it (never in a run whose result bundle is complete, which is immutable)."
        )
    # A complete bundle is immutable (docs/RESULT_BUNDLES.md). The resolve cell refused every session it could
    # predict would judge or train into one; this catches the rest (a node held only as an ancestors/ record that
    # TRUNK_DIR no longer certifies) before the node's first write.
    refuse_write_into_complete_run(RUN_DIR, what=f"Judging or training {NODE.id!r}")
    if not covered and verdict is None and Path(f"{final_stem}.zip").exists() and Path(final_vecnorm).exists():
        # (3) JUDGE
        print(f"Judging {NODE.id!r}: {stage_dir} holds trained checkpoints but no gate verdict.")
        model, handoff_stem, final_model_path, handoff_vecnorm, results = evaluate_stage_checkpoints(
            SPECIES_CFG,
            STAGE_CONFIGS[stage],
            stage,
            ALGORITHM,
            stage_dir,
            final_path=final_stem,
            final_vecnorm_path=final_vecnorm,
            timesteps=budget,
            duration_seconds=read_stage_duration(stage_dir) or 0.0,
            plant_identity=PLANT_IDENTITY,
            evaluation_seed=EVALUATION_SEED,
        )
    elif not covered and verdict is None and any(model_dir.glob("*.zip")):
        raise RuntimeError(
            f"{stage_dir} holds periodic checkpoints but no completed stage ({final_stem.name}.zip is missing): an "
            f"interrupted node. Set RESUME_STAGE = {stage!r} in the RESUME cell (section 5) and run all again: it "
            "finishes the budget, then this loop judges the node (a budget its periodic checkpoints already spent "
            "without an intact final pair is refused there: that node trains again in a fresh RUN_ID). It is never "
            "retrained from scratch over them."
        )
    else:
        # (2) TRAIN
        if parent_handoff is not None:
            load_path, vecnorm_path, task_load_mode = parent_handoff["model"], parent_handoff["vecnorm"], "initialize_next_stage"
        else:
            load_path, vecnorm_path, task_load_mode = None, None, "resume_same_stage"
        # PRE-REGISTRATION for a frozen-null gate kind (recovery): thresholds and
        # null panels freeze BEFORE the policy trains, so it is never judged
        # against a moving target. The brace null is the parent's handoff held
        # at its post-settle mean action, so a root has nothing to freeze from.
        if gate_kind in FROZEN_NULL_GATE_KINDS and parent_handoff is None:
            raise RuntimeError(
                f"{NODE.id!r} is judged by the frozen-null gate {gate_kind}, whose brace null needs a parent "
                "checkpoint, but it is a root of the manifest"
            )
        if gate_kind in FROZEN_NULL_GATE_KINDS and not (stage_dir / "gate_resolution.json").exists():
            print(f"Freezing the {gate_kind} resolution for {NODE.id!r} (statue + brace null panels)...")
            freeze_recovery_gate(
                stage_dir,
                species=SPECIES,
                stage=stage,
                policy_zip=f"{parent_handoff['model']}.zip",
                vecnorm=parent_handoff["vecnorm"],
                algorithm=ALGORITHM,
            )
        if gate_kind in FROZEN_NULL_GATE_KINDS:
            # Refuse stale calibrations or abbreviated rehearsal panels before
            # spending the training budget.
            validate_recovery_resolution(
                stage_dir,
                species=SPECIES,
                stage=stage,
                policy_zip=f"{parent_handoff['model']}.zip",
                vecnorm=parent_handoff["vecnorm"],
                algorithm=ALGORITHM,
            )
        model, handoff_stem, final_model_path, _stage_dir, handoff_vecnorm, results = train_stage(
            stage=stage,
            timesteps=budget,
            run_dir=RUN_DIR,
            load_path=load_path,
            vecnorm_path=vecnorm_path,
            task_load_mode=task_load_mode,
            parent_run_id=parent_handoff["run_id"] if parent_handoff is not None else None,
            label=RUN_LABEL or None,
        )

    # A frozen-null kind rolls the trained policy over EXACTLY the frozen panel:
    # same seeds, same push schedules, same calibrated judge as the frozen
    # nulls. A missing, tampered or stale-task resolution refuses here.
    panel_successes = None
    if gate_kind in FROZEN_NULL_GATE_KINDS:
        panel_evidence = roll_policy_panel(
            stage_dir, f"{handoff_stem}.zip", handoff_vecnorm, species=SPECIES, stage=stage, algorithm=ALGORITHM
        )
        write_recovery_evidence(stage_dir, panel_evidence)
        panel_successes = panel_evidence.successes_by_seed()
        print(f"Policy panel on the frozen seeds: {sum(panel_successes.values())}/{len(panel_successes)} episodes recovered")

    # Artifacts: summary, videos, graphs — and the gate verdict, judged through
    # the one shared reporting.gates.evaluate_stage_gate, recorded onto the
    # results dict and written to gate_verdict.json (what reuse reads later).
    results = generate_stage_artifacts(
        species_cfg=SPECIES_CFG,
        stage_config=config,
        stage=stage,
        algorithm=ALGORITHM,
        stage_dir=stage_dir,
        seed=SEED,
        stage_results=results,
        recovery_successes_by_seed=panel_successes,
    )
    NODE_RESULTS[NODE.id] = results
    completed_stages.append((stage, stage_dir))
    display_stage_videos(stage_dir)
    plot_training_curves([(stage, stage_dir)], STAGE_CONFIGS, ALGORITHM)
    plot_diagnostics_graphs([(stage, stage_dir)], STAGE_CONFIGS, ALGORITHM)

    # Summary and bundle BEFORE the gate check, so every certified deliverable
    # above a failure is published (result schema v4: the bundle is `partial`).
    write_training_summary(RUN_DIR, chain_results())
    save_run_bundle(chain_results(), species=SPECIES)

    # Enforce the recorded verdict. halt releases the runtime, then raises: the raise
    # stops "Run all", so the auto-disconnect cell at the end would never execute.
    if not results["publication_gate_passed"]:
        _gate_msg = f"{NODE.id} failed its curriculum gate: " + "; ".join(results["gate_failures"]) + "."
        halt(_gate_msg, in_colab=IN_COLAB, auto=AUTO_DISCONNECT, flush_drive=USE_GOOGLE_DRIVE)
    print(f"{NODE.id}: gate PASSED ({gate_kind}); handoff {handoff_stem}.zip")
    NODE_HANDOFF[NODE.id] = {
        "model": handoff_stem,
        "vecnorm": handoff_vecnorm,
        "stage_dir": stage_dir,
        "run_dir": RUN_DIR,
        "run_id": None,
        "model_sha256": sha256_file(f"{handoff_stem}.zip"),
        "normalization_sha256": sha256_file(handoff_vecnorm),
        "reused": False,
    }

## 7. Manual single node (debugging escape hatch)

Train ONE node outside the chain — to probe a config, or to rerun a node from an arbitrary checkpoint — by setting `MANUAL_NODE` below. The load's edge is declared, never inferred: `MANUAL_LOAD_MODE = "initialize_next_stage"` for a parent's handoff, `"resume_same_stage"` for this node's own checkpoint. The node's artifacts and verdict are generated and recorded into the summary and bundle exactly as the chain does, but the verdict is **never enforced** here (no raise, no disconnect) and the node never enters the chain's handoff table: nothing downstream trains on it. A run whose bundle is `complete` takes no manual node: the cell refuses before it writes anything, so probe in a fresh `RUN_ID`. Leave `MANUAL_NODE = None` for an unattended Run-all.

In [ ]:
# ===== MANUAL SINGLE NODE — debugging escape hatch =====
MANUAL_NODE = None  # None = skip; a stage reference (2 or "recovery") to train ONE node outside the chain
MANUAL_LOAD_PATH = None  # checkpoint stem to load (None = from scratch); its sidecar goes in MANUAL_VECNORM_PATH
MANUAL_VECNORM_PATH = None
MANUAL_LOAD_MODE = "initialize_next_stage"  # the load's declared edge: a parent's handoff, or "resume_same_stage"
MANUAL_TIMESTEPS = None  # None = the node's TOML budget (QUICK_TEST shortens it)

if MANUAL_NODE is not None:
    from environments.shared.curriculum import FROZEN_NULL_GATE_KINDS
    from environments.shared.harnesses.freeze_recovery_gate import (
        freeze_recovery_gate,
        roll_policy_panel,
        validate_recovery_resolution,
    )
    from environments.shared.recovery_evaluation import write_recovery_evidence
    from environments.shared.result_bundle import ResultBundleError, refuse_write_into_complete_run

    _manual_entry = MANIFEST.resolve(MANUAL_NODE)
    # A complete bundle is immutable: refused here, before the freeze or the training below writes anything.
    refuse_write_into_complete_run(RUN_DIR, what=f"The manual node {_manual_entry.id!r}")
    _manual_stage = _manual_entry.reference
    _manual_config = STAGE_CONFIGS[_manual_stage]
    _manual_dir = Path(RUN_DIR) / stage_dirname(SPECIES, _manual_stage)
    _manual_gate_kind = _manual_config.get("curriculum_kwargs", {}).get("gate_kind")
    _manual_budget = MANUAL_TIMESTEPS or (50_000 if QUICK_TEST else _manual_config["curriculum_kwargs"]["timesteps"])
    _manual_policy_zip = f"{MANUAL_LOAD_PATH}.zip" if MANUAL_LOAD_PATH else None

    # A frozen-null kind (recovery) pre-registers its gate before training,
    # exactly as the chain loop does.
    if _manual_gate_kind in FROZEN_NULL_GATE_KINDS and not (_manual_dir / "gate_resolution.json").exists():
        print(f"Freezing the {_manual_gate_kind} resolution for {_manual_entry.id!r}...")
        freeze_recovery_gate(
            _manual_dir,
            species=SPECIES,
            stage=_manual_stage,
            policy_zip=_manual_policy_zip,
            vecnorm=MANUAL_VECNORM_PATH,
            algorithm=ALGORITHM,
        )
    if _manual_gate_kind in FROZEN_NULL_GATE_KINDS:
        validate_recovery_resolution(
            _manual_dir,
            species=SPECIES,
            stage=_manual_stage,
            policy_zip=_manual_policy_zip,
            vecnorm=MANUAL_VECNORM_PATH,
            algorithm=ALGORITHM,
        )

    _manual_model, _manual_handoff, _manual_final, _manual_dir, _manual_vecnorm, _manual_results = train_stage(
        stage=_manual_stage,
        timesteps=_manual_budget,
        run_dir=RUN_DIR,
        load_path=MANUAL_LOAD_PATH,
        vecnorm_path=MANUAL_VECNORM_PATH,
        task_load_mode=MANUAL_LOAD_MODE if MANUAL_LOAD_PATH else "resume_same_stage",
        label=RUN_LABEL or None,
    )

    _manual_panel = None
    if _manual_gate_kind in FROZEN_NULL_GATE_KINDS:
        _manual_evidence = roll_policy_panel(
            _manual_dir,
            f"{_manual_handoff}.zip",
            _manual_vecnorm,
            species=SPECIES,
            stage=_manual_stage,
            algorithm=ALGORITHM,
        )
        write_recovery_evidence(_manual_dir, _manual_evidence)
        _manual_panel = _manual_evidence.successes_by_seed()

    _manual_results = generate_stage_artifacts(
        species_cfg=SPECIES_CFG,
        stage_config=_manual_config,
        stage=_manual_stage,
        algorithm=ALGORITHM,
        stage_dir=_manual_dir,
        seed=SEED,
        stage_results=_manual_results,
        recovery_successes_by_seed=_manual_panel,
    )
    NODE_RESULTS[_manual_entry.id] = _manual_results
    completed_stages.append((_manual_stage, _manual_dir))
    display_stage_videos(_manual_dir)

    # Recorded, never enforced: this cell is a probe, not a chain step, so no
    # raise and no disconnect — and it never feeds NODE_HANDOFF.
    if _manual_results["publication_gate_passed"]:
        print(f"{_manual_entry.id}: gate verdict PASS ({_manual_gate_kind})")
    else:
        print(f"{_manual_entry.id}: gate verdict FAIL ({_manual_gate_kind}) —", "; ".join(_manual_results["gate_failures"]))

    try:
        write_training_summary(RUN_DIR, chain_results())
        save_run_bundle(chain_results(), species=SPECIES)
    except ResultBundleError as exc:
        # Never swallowed silently (D-A11): a manual node that the bundle
        # cannot place (e.g. its parent is not in this run) is reported.
        print(f"Result bundle not rewritten for the manual node: {exc}")

## 8. Report

Evaluate the behavior's deliverable, show this run's training curves, replay the chain's videos and verify the result bundle.

### Evaluate the behavior's policy

In [ ]:
# Evaluate BEHAVIOR's deliverable — the target node's certified handoff checkpoint —
# using the library's evaluate() function, which provides full locomotion metrics
# (gait symmetry, cost of transport, stride frequency, velocity consistency, etc.)
# via LocomotionMetrics.
if TARGET_NODE.id not in NODE_HANDOFF:
    raise RuntimeError(
        f"{TARGET_NODE.id!r} (behavior {BEHAVIOR!r}) is not certified in this session; run the chain loop "
        "(section 6) to the end first"
    )
print(f"Evaluating the {BEHAVIOR!r} policy — {TARGET_NODE.id} ({ALGORITHM})...")
evaluate(
    species_cfg=SPECIES_CFG,
    stage_configs=STAGE_CONFIGS,
    model_path=NODE_HANDOFF[TARGET_NODE.id]["model"] + ".zip",
    n_episodes=30,
    render=False,
    stage=TARGET_NODE.reference,
    algorithm=ALGORITHM.lower(),
)

### Training curves

In [ ]:
# Display the graphs of every node trained or judged in THIS run (convenient for
# re-running this cell standalone). Reused ancestors keep their curves in the run
# that trained them. Display only: each node's saved copies are the figures/ set
# the chain loop wrote and declared before sealing the bundle, and a file written
# here would be undeclared, so the cleanup cell's bundle check would raise and
# "Run all" would stop before the auto-disconnect.
for stage_ref, stage_dir in completed_stages:
    print(f"\n{'=' * 40}")
    print(f"{MANIFEST.resolve(stage_ref).id}: {STAGE_CONFIGS[stage_ref]['name']}")
    print(f"{'=' * 40}")
    plot_training_curves([(stage_ref, stage_dir)], STAGE_CONFIGS, ALGORITHM)
    plot_diagnostics_graphs([(stage_ref, stage_dir)], STAGE_CONFIGS, ALGORITHM)

### Replay the chain's videos

Display the saved videos for every node of the behavior's chain — a reused
ancestor plays from the run that certified it. Videos are recorded by
`generate_stage_artifacts` after each node completes.

In [ ]:
for entry in CHAIN:
    handoff = NODE_HANDOFF.get(entry.id)
    print(f"\n{'=' * 40}")
    if handoff is None:
        print(f"{entry.id}: not certified in this session (no videos)")
        continue
    origin = f"reused from run {handoff['run_id']}" if handoff["run_id"] else "this run"
    print(f"{entry.id}: {STAGE_CONFIGS[entry.reference]['name']} ({origin})")
    print(f"{'=' * 40}")
    display_stage_videos(handoff["stage_dir"])

### Verify the result bundle

In [ ]:
# Verify the bundle without rewriting immutable artifacts. Every deliverable
# this run certified is published (result schema v4); the bundle is
# `complete` only when its target — BEHAVIOR's node — and every recorded
# deliverable are certified. A chain that stopped above its target (or a
# manual node that failed its gate) leaves the bundle publishable but
# `partial`: report which deliverable is uncertified instead of failing.
from environments.shared.result_bundle import ResultBundleError

try:
    bundle_report = validate_result_bundle(RUN_DIR, require_complete=True)
except ResultBundleError as incomplete:
    bundle_report = validate_result_bundle(RUN_DIR, require_complete=False, require_publishable=True)
    _uncertified = sorted(key for key, record in bundle_report["deliverables"].items() if not record["certified"])
    print(f"Result bundle is publishable but not complete ({bundle_report['status']}): uncertified {_uncertified}")
    print(f"  {incomplete}")
print(f"Result bundle verified: {bundle_report['status']}")

print("Training complete!")
print(f"\nBehavior: {BEHAVIOR} -> {TARGET_NODE.id}")
print(f"Algorithm: {ALGORITHM}")
print(f"Run directory: {RUN_DIR}")
for entry in CHAIN:
    handoff = NODE_HANDOFF.get(entry.id)
    if handoff is None:
        print(f"  {entry.id}: NOT certified in this session")
    elif handoff["run_id"]:
        print(f"  {entry.id}: reused from run {handoff['run_id']} — {handoff['model']}.zip")
    else:
        print(f"  {entry.id}: certified in this run — {handoff['model']}.zip")
print("\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")

## 9. Auto-Disconnect

Releases the Colab runtime when the behavior chain finishes so it doesn't sit idle burning GPU credits. A node that fails its gate disconnects the same way from within the chain loop (the gate's `RuntimeError` halts "Run all" before this cell). Set `AUTO_DISCONNECT = False` in the configuration cell to keep the runtime alive for interactive work.

In [ ]:
disconnect_runtime(
    f"Training finished — behavior {BEHAVIOR!r} ({TARGET_NODE.id}) chain complete.",
    in_colab=IN_COLAB,
    auto=AUTO_DISCONNECT,
    flush_drive=USE_GOOGLE_DRIVE,
)